In [ ]:
import os
import re
import sys
import copy
import json
import torch
import spacy
import pickle
import contextlib
import numpy as np
from tqdm.auto import tqdm
from typing import List, Tuple, Dict
import logging

from qiskit_aer import AerSimulator
from pytket.extensions.qiskit.backends.aer import AerBackend
# from qiskit.providers.aer import AerSimulator
# from pytket.extensions.qiskit import AerBackend


from lambeq.backend.grammar import Diagram
from lambeq import (
    AtomicType,
    IQPAnsatz,
    RemoveCupsRewriter,
    Rewriter,
    UnifyCodomainRewriter,
    DepCCGParser
)

from collections.abc import Mapping, Sequence
import statistics

from pathlib import Path
import traceback
import gc

# import depccg
# from lambeq import ( CCGParser, CCGTree, CCGRuleUseError, CCGRule, CCGType,
#                     CCGBankParseError, CCGBankParser, DepCCGParseError )


In [ ]:
@contextlib.contextmanager
def suppress_all_output():
    """
    Suppresses Python stdout/stderr and low-level FD stdout/stderr.
    This also catches many C/C++/CUDA/subprocess writes.
    """
    logging.disable(logging.CRITICAL)

    # Flush current streams
    sys.stdout.flush()
    sys.stderr.flush()

    # Save original file descriptors
    old_stdout_fd = os.dup(1)
    old_stderr_fd = os.dup(2)

    try:
        with open(os.devnull, "w") as devnull:
            devnull_fd = devnull.fileno()

            # Redirect low-level stdout/stderr
            os.dup2(devnull_fd, 1)
            os.dup2(devnull_fd, 2)

            # Redirect Python-level stdout/stderr too
            with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
                yield

    finally:
        # Restore stdout/stderr
        os.dup2(old_stdout_fd, 1)
        os.dup2(old_stderr_fd, 2)

        os.close(old_stdout_fd)
        os.close(old_stderr_fd)

        logging.disable(logging.NOTSET)

In [ ]:
import logging
logging.getLogger("allennlp").setLevel(logging.WARNING)
logging.getLogger("depccg").setLevel(logging.WARNING)
parser = DepCCGParser(model='elmo', device=0) # device=:  -1 == CPU | 0 == GPU | 1 == second GPU

In [ ]:
# https://chatgpt.com/s/t_69b2c7f7e28081919384bb842e7c46b3 - apie sakiniu preprocesinga kuris sutrumpina 40-60 proc sakiniu ilgio

ansatz    = IQPAnsatz(
    {AtomicType.SENTENCE: 1,
     AtomicType.NOUN:     1, # Galima priskirti 2 qubitus, jei, pvz, treniravimo rezultatai yra prasti
     }, # AtomicType.PREPOSITIONAL_PHRASE: 0,
    n_layers=1, n_single_qubit_params=3
)

def create_rewriter():
    # Rule to delete conjunction boxes (“and”, “but”) # just the wire, no box
    # conj_rule = SimpleRewriteRule(cod=AtomicType.CONJUNCTION, template=Id(AtomicType.CONJUNCTION))

    rewriter = Rewriter([
        'determiner',
        'auxiliary', # potentially risky. jei tai nepasalina daug qubitu, tai ismest
        'connector',
        # 'coordination',
        'prepositional_phrase', 
        # 'subject_rel_pronoun', # They don't hurt performance, and they act as a safety
    ])
    
    # rewriter.add_rules(conj_rule)
    return rewriter

rewriter = create_rewriter()
remove_cups = RemoveCupsRewriter()
unify = UnifyCodomainRewriter(output_type=AtomicType.SENTENCE)

simulator = AerSimulator(
    method="statevector", device="GPU",
    precision="single",         # 32-bit float for ~2× speedup on large statevectors
    cuStateVec_enable=True  ,     # turn on NVIDIA cuStateVec kernels
    # batched_shots_gpu=True,     # batch thousands of shots very efficiently on GPU
    # batched_shots_gpu_max_qubits=29, # 8 ⋅ 2^n == 7 ⋅ 2^30, n ~= 29.8
    # num_threads_per_device=2    # limit CPU threads per GPU to reduce overhead
)

backend = AerBackend(simulation_method="statevector")
backend._qiskit_backend = simulator

# comp_pass = backend.default_compilation_pass(2)

In [ ]:
def get_deep_type(obj):
    if isinstance(obj, list):
        # We look at the unique types inside the list to keep it readable
        inner_types = {get_deep_type(item) for item in obj}
        return f"List[{' | '.join(sorted(inner_types))}]"

    elif isinstance(obj, dict):
        # We summarize the types of all keys and all values
        key_types = {get_deep_type(k) for k in obj.keys()}
        val_types = {get_deep_type(v) for v in obj.values()}
        return f"Dict[{' | '.join(sorted(key_types))}, {' | '.join(sorted(val_types))}]"

    else:
        # Return the class name (e.g., 'Diagram' or 'str')
        return type(obj).__name__

def get_deep_shape(obj, level=0):
    indent = "  " * level
    
    # 1. Atomic types (Strings/Bytes) - Check these first 
    # because they are technically Sequences too!
    if isinstance(obj, (str, bytes)):
        return f"{indent}str"

    # 2. Dictionaries (Mappings)
    elif isinstance(obj, Mapping):
        if not obj:
            return f"{indent}dict(len=0)"

        lines = [f"{indent}dict(len={len(obj)})"]
        for key, value in obj.items():
            # Get child shape and strip only the first line's indentation 
            # so we can prefix it with our key label
            child = get_deep_shape(value, level + 1).lstrip()
            lines.append(f"{indent}  key={repr(key)} -> {child}")
        return "\n".join(lines)

    # 3. Sequences (Lists, Tuples, etc.)
    elif isinstance(obj, Sequence):
        name = type(obj).__name__
        if not obj:
            return f"{indent}{name}(len=0)"

        header = f"{indent}{name}(len={len(obj)})"
        
        # Calculate shapes of all children
        child_shapes = [get_deep_shape(item, level + 1).lstrip() for item in obj]
        unique_shapes = sorted(list(set(child_shapes)))

        if len(unique_shapes) == 1:
            # All items are identical structure
            return f"{header}\n{indent}  [*] -> {unique_shapes[0]}"
        else:
            lines = [header]
            for i, shape in enumerate(child_shapes):
                lines.append(f"{indent}  [{i}] -> {shape}")
            return "\n".join(lines)

    # 4. Base objects (int, float, None, etc.)
    else:
        return f"{indent}{type(obj).__name__}"

def find_mismatches(data_a: List, data_b: List):
    # 1. Check if the outer lists are even the same length
    if len(data_a) != len(data_b):
        print(
            f"❌ [CRITICAL] Outer List Length Mismatch: List A={len(data_a)}, List B={len(data_b)}"
        )

    # Iterate through the top-level list
    for i, (dict_a, dict_b) in enumerate(zip(data_a, data_b)):
        # Check if the keys in the dictionaries match
        keys_a = set(dict_a.keys())
        keys_b = set(dict_b.keys())

        if keys_a != keys_b:
            print(f"❌ [Index {i}] Key Mismatch:")
            print(f"   Keys only in A: {keys_a - keys_b}")
            print(f"   Keys only in B: {keys_b - keys_a}")
            continue  # Skip to next list item if keys don't match

        # Check values for each key
        for key in keys_a:
            list_a = dict_a[key]
            list_b = dict_b[key]

            # 2. Check lengths of the lists inside the dictionary
            if len(list_a) != len(list_b):
                print(f"❌ [Index {i}][Key: '{key}'] Inner List Length Mismatch:")
                print(f"   Length A: {len(list_a)}")
                print(f"   Length B: {len(list_b)}")

            # 3. Check individual elements inside those lists
            # This handles List[Diagram], List[List[int]], and List[str]
            for j, (val_a, val_b) in enumerate(zip(list_a, list_b)):
                if val_a != val_b:
                    print(
                        f"❌ [Index {i}][Key: '{key}'][Inner Index {j}] Content Mismatch!"
                    )
                    print(f"   Type A: {type(val_a).__name__}")
                    print(f"   Type B: {type(val_b).__name__}")

                    # If they are small (like List[int] or str), print the actual value
                    if not hasattr(
                        val_a, "draw"
                    ):  # Don't print full Diagrams, they are too big
                        print(f"   Value A: {val_a}")
                        print(f"   Value B: {val_b}")
                    else:
                        print(
                            f"   (Diagram content differs - possibly different boxes or wires)"
                        )

def diagnose_variable(variable):
    # print("TOP LEVEL LENGTH:", len(variable) if hasattr(variable, '__len__') else "N/A")
    print("DEEP TYPE:")
    print(get_deep_type(variable))
    print("\nDEEP SHAPE:")
    print(get_deep_shape(variable))

def print_n_qubits(n_qubits_list: List[int]):
    print(f"qubit_list: {n_qubits_list}")
    print(f"qubit_list_length: {len(n_qubits_list)}")
    print(f"qubit_list_mean: {statistics.mean(n_qubits_list)}\n")
    


In [ ]:
CLEAN_REGEX = re.compile(r"[^\w\s'-]")

# https://chatgpt.com/s/t_69c6b7d3545c8191a4935319498eee59  -> shows and explains how to use spacy to remove `appos`, `acl`, `advcl`, etc. relations
REMOVE_DEPS = {
    "appos",  # appositive
    "acl",    # clausal modifier
    "advcl",  # adverbial clause
    "relcl",  # relative clause
    # "amod",   # Adjectival Modifier (optional if the circuits are still to big)
    # "xcomp",   # open clausal complement | Probably do not use
    # "ccomp"    # clausal complement | Probably do not use

}

# https://chatgpt.com/s/t_69cd50d9e7848191aca0c6ccd42a1ff8 -> expains "ner", "textcat", "lemmatizer", "tagger", "parser"
nlp = spacy.load(
    # "en_core_web_sm",
    "en_core_web_trf",
    disable=["ner", "textcat", "lemmatizer", "tagger"],  # keep only parser
)

def sentence_simplify_spacy(sentences: List[str]) -> Tuple[List[str], List[int]]:
    sentences_simplified = []
    removed_idx = []

    for idx, doc in enumerate(nlp.pipe(sentences, batch_size=128)):
        to_remove = set()

        # Step 1: Identify subordinate clauses to remove
        for token in doc:
            if token.dep_ in REMOVE_DEPS:
                to_remove.update(t.i for t in token.subtree)

        # Step 2: Rebuild sentence
        pruned = "".join(
            token.text_with_ws
            for token in doc
            if token.i not in to_remove
        )

        # Step 3: Clear symbols
        cleaned = clean_sentence(pruned)
        # Step 4: Remove multi-spaces
        cleaned = " ".join(cleaned.split())

        if cleaned:
            sentences_simplified.append(cleaned)
        else:
            removed_idx.append(idx)

    print(f"In pruning lost {len(removed_idx)} out of {len(sentences)} sentences.")
    return sentences_simplified, removed_idx

def clean_sentence(sent: str) -> str:
    return CLEAN_REGEX.sub("", sent)

def remove_negative_articles(dataset: List[Dict], sentences_key: str, labels_key: str,
    positive_label, min_positive_ratio: float = 0.10):
    invalid_ratio, valid_articles = [], []
    one_positive_low_ratio, two_positive_low_ratio = [], []
    zero_positive = []
    label_distribution = {
        1: 0,
        2: 0,
        3: 0,
        4: 0,
        ">4": 0
    }

    for i, article in enumerate(dataset):
        sentences = article.get(sentences_key)
        labels = article.get(labels_key)

        if sentences is None or labels is None:
            invalid_ratio.append(i)
            continue

        n_labels = len(labels)

        if len(sentences) != len(labels):
            invalid_ratio.append(i)
            continue

        if n_labels == 0:
            invalid_ratio.append(i)
            continue

        positive_count = labels.count(positive_label)
        positive_ratio = positive_count / n_labels

        if positive_count == 0:
            zero_positive.append(i)
            continue

        elif positive_count in label_distribution:
            label_distribution[positive_count] += 1
        else:
            label_distribution[">4"] += 1
        

        if n_labels > 12 and positive_count == 1:
            one_positive_low_ratio.append(i)
        elif n_labels > 20 and positive_count == 1:
            two_positive_low_ratio.append(i)

        valid_articles.append(article)

    print(f"No+: {len(zero_positive)} | 1+ (>12): {len(one_positive_low_ratio)} | 2+ (>30): {len(two_positive_low_ratio)}")
    print(f"Label distribution: 1: {label_distribution[1]} | 2: {label_distribution[2]} | 3: {label_distribution[3]} | 4: {label_distribution[4]} | >4: {label_distribution['>4']}")
    print(f"Positive labels ratio: {positive_ratio:.4f}")
    print(f"Invalid ratio: {len(invalid_ratio)} out of {len(dataset)} articles.")

    return valid_articles

def load_PreSumm_pts(file_number: int=0, ds_purpose: str="train", calculate_articles: bool=False) -> List[Dict] | Tuple[List[Dict], List[int], List[int], List[int]]:
    valid_ds, elements_valid_labels = [], []
    n_sentences, n_labels, n_labels_true = [], [], []
    
    print(f"load_PreSumm_pts_{file_number}")
    file_path = f"Dataset/Raw/cnn_dailymail/_PreSumm/cnndm.{ds_purpose}.{file_number}.bert.pt"
    loaded_data = torch.load(file_path)
    
    elements_valid_labels = remove_negative_articles(loaded_data, "src_txt", "src_sent_labels", 1)

    for i, file_article in enumerate(elements_valid_labels):
        quantum_state_distribution_labels = []
        for label in file_article["src_sent_labels"]:
            if label:
                quantum_state_distribution_labels.append([0,1])
            else:
                quantum_state_distribution_labels.append([1,0])

        valid_ds.append(
            {
                "text_sentences": file_article["src_txt"],
                # "org_labels": line["src_sent_labels"],
                "labels": quantum_state_distribution_labels
            }
        )
        n_sentences.append(len(file_article["src_txt"]))
        n_labels.append(len(file_article["src_sent_labels"]))
        n_labels_true.append(file_article["src_sent_labels"].count(1))

    if calculate_articles:
        n_sentences_sum = sum(n_sentences)
        labes_sum       = sum(n_labels)
        labels_true_sum = sum(n_labels_true)
        print(f"\n n_sentences: {n_sentences_sum} \n n_labels: {labes_sum} (average {labes_sum/len(n_labels):.4f}) \n n_labels_true: {labels_true_sum} (average {labels_true_sum/len(n_labels_true):.4f})")
    
    return valid_ds

def load_encoded(filename = "Dataset/Encoded/cnn_dailymail/PreSumm_0_701.pkl"):
    with open(filename, 'rb') as file:
        loaded_encoded_ds = pickle.load(file)

    return loaded_encoded_ds



In [ ]:
def remove_by_idx(ls: List, remove: List[int]):
    remove_set = set(remove)
    return [x for i, x in enumerate(ls) if i not in remove_set]

def sent2diagrams(sentences: List[str]):
    none_idx = []
    valid_diagrams = []

    diagrams = parser.sentences2diagrams(
        sentences, tokenised=False, suppress_exceptions=True
    )

    for i, diag in enumerate(diagrams):
        if diag is None:
            none_idx.append(i)
        else:
            valid_diagrams.append(diag)

    print(f"In sent2diagrams() lost {len(none_idx)} out of {len(sentences)}.")

    return valid_diagrams, none_idx

def normalize(sentence_diagrams: List[Diagram]):
    diagrams_normalized, none_idx, errs = [], [], []
    drop_rewrite = 0
    drop_exception  = 0
    for i, d in enumerate(sentence_diagrams):
        try:
            d = rewriter(d)
            if d is None:
                none_idx.append(i)
                drop_rewrite += 1
                continue

            d = remove_cups(d)
            d = d.normal_form()
            # d = d.pregroup_normal_form()
            d = unify(d)
            d = d.normal_form()

            if d.cod != AtomicType.SENTENCE:
                raise RuntimeError(f"Unexpected codomain: {d.cod}. It should be AtomicType.SENTENCE")

        except Exception as e:
            none_idx.append(i)
            errs.append(f"normalize() | {type(e).__name__}: {e}")
            drop_exception  += 1
            continue

        diagrams_normalized.append(d)

    print(f"In normalize() lost {drop_rewrite} (rewrite) and {drop_exception } (cup removal) out of {len(sentence_diagrams)}.")

    return diagrams_normalized, none_idx, errs

def quantum_encode(diagrams: List[Diagram]):
    encoded_diagrams, remove, errs = [], [], []
    for i, diagram in enumerate(diagrams):
        try:
            circ = ansatz(diagram)
            encoded_diagrams.append(circ)
        except Exception as e:
            errs.append(f"quantum_encode() | {type(e).__name__}: {e}")
            remove.append(i)

    print(f"In quantum_encode() lost {len(remove)} out of {len(diagrams)}.")
    print(len(errs))

    return encoded_diagrams, remove, errs

def will_train(
    circuits: List,
    qubit_limit: int=26,
    mem_limit_bytes: int=7 * 2**30,
    circuit_depth_limit: int=80,
    gate_limit: int=3000,
):

    valid, invalid_idxs, errs = [], [], []
    n_qubits_list = []

    for idx, circ in enumerate(circuits):
        try:
            tk_circ = circ.to_tk()
            n_qubits = tk_circ.n_qubits
            n_gates = tk_circ.n_gates
            depth = tk_circ.depth()


            bytes_per_amplitude = 8  # complex64: 2 × float32
            needed_bytes = bytes_per_amplitude * (2 ** n_qubits)
            if n_qubits > qubit_limit:
                raise RuntimeError(f"Too many qubits: {n_qubits} > {qubit_limit}")

            if needed_bytes > mem_limit_bytes:
                raise RuntimeError(
                    f"Needs {needed_bytes / 2**30:.2f} GiB > limit {mem_limit_bytes / 2**30:.2f} GiB"   
                )
            
            # even when samll amount of qubits, there could be a lot of gates and circuit depth
            # which will slow down training process
            if depth > circuit_depth_limit:
                raise RuntimeError(f"Circuit too deep: {depth}")

            if n_gates > gate_limit:
                raise RuntimeError(f"Circuit contains too many gates: {n_gates}")

            # compiled = backend.get_compiled_circuit(tk_circ.copy())
            # comp_pass = backend.default_compilation_pass(2)
            # compiled = comp_pass.apply(tk_circ.copy())

            # depending on pytket version, .apply() mutates and may return True/False, not the circuit
            # tk_circ_compiled = tk_circ.copy()
            # comp_pass.apply(tk_circ_compiled)

            # if compilation_pass is not None:
            #     tk_circ_compiled = tk_circ.copy()
            #     compilation_pass.apply(tk_circ_compiled)
            #     tk_circ = tk_circ_compiled
            n_qubits_list.append(n_qubits)

        except Exception as e:
            invalid_idxs.append(idx)
            errs.append(f"will_train() | {type(e).__name__}: {e}")
            continue

        valid.append(circ)

    
    print(f"In will_train() lost {len(invalid_idxs)} out of {len(circuits)}.")

    return valid, invalid_idxs, errs, n_qubits_list

def preprocess_and_encode(dataset: List[Dict]) -> Tuple[List[Dict], List[str], List[int]]:
    print(f"Preprocessing and encoding {len(dataset)} articles...")
    encoded_data, errors, all_n_qubits = [], [], []

    # for i, data_dict in enumerate(tqdm(dataset, desc="Filtering and Encoding dataset")):
    for i, data_dict in enumerate(dataset):
        # with suppress_all_output():
            # clear_output(wait=False)
            text_sentences = data_dict["text_sentences"].copy()
            labels         = copy.deepcopy(data_dict["labels"])

            n_text_sentences = len(text_sentences)
            n_labels = len(labels)

            sentences_simplified, remove = sentence_simplify_spacy(text_sentences)
            text_sentences               = remove_by_idx(text_sentences, remove)
            labels                       = remove_by_idx(labels, remove)

            diagrams, remove = sent2diagrams(sentences_simplified)
            text_sentences   = remove_by_idx(text_sentences, remove)
            labels           = remove_by_idx(labels, remove)

            normalized_diagrams, remove, errs2 = normalize(diagrams)
            text_sentences                     = remove_by_idx(text_sentences, remove)
            labels                             = remove_by_idx(labels, remove)


            circuits, remove, errs3 = quantum_encode(normalized_diagrams)
            text_sentences          = remove_by_idx(text_sentences, remove)
            labels                  = remove_by_idx(labels, remove)

            circuits, remove, errs4, n_qubits_list = will_train(circuits)
            text_sentences                         = remove_by_idx(text_sentences, remove)
            labels                                 = remove_by_idx(labels, remove)

            print("init:", n_text_sentences, n_labels,"\nafter:", len(circuits), len(text_sentences), len(labels))

            if len(circuits) == 0:
                errors.append("preprocess_and_encode() | no valid circuits after filtering")
                continue

            encoded_data.append(
                {
                    "article_id": data_dict["article_id"],
                    "circuits": circuits,
                    "labels": labels,
                    "n_qubits": n_qubits_list,
                    "original_text_sentences": text_sentences,
                }
            )

            errors += errs2 + errs3 + errs4 # + errs1 sent2diagrams has suppress_exceptions=True, so instead of errors, it returns None
            all_n_qubits.append(n_qubits_list)

    # errors.append(f"n_sentences: {n_text_sentences} | n_circuits: {len(circuits)} | {n_text_sentences - len(circuits)}")

    return encoded_data, errors, all_n_qubits



In [ ]:
FILE_PATH = "Dataset/Raw/WikiHow/Labeled_wikihowSep_200k.jsonl"

def load_WikiHow_JSON(file_path: str, n_elements_to_read: int=0):
    if not n_elements_to_read:
        n_elements_to_read = sys.maxsize # sys.maxint has been removed in Python 3. Use sys.maxsize as a practical alternative.
    dataset = []
    with open(file_path, "r", encoding="utf-8") as f:

        for i, line in enumerate(f):
            if i == n_elements_to_read:
                break
            record = json.loads(line)
            dataset.append(record)

    return dataset

def extract_relevant_data(dataset):
    extracted_ds = []
    for i, article in enumerate(dataset):
        sents, labels = zip(*[(sent_dict["sentence"], sent_dict["label_vector"])
            for sent_dict in article["sentences_data"]
        ])
        
        article_record = {
            "article_id": article["article_id"],
            "text_sentences": list(sents),
            "labels": list(labels),
        }
        extracted_ds.append(article_record)
    
    return extracted_ds



In [ ]:
def encode_auto(dataset: List[Dict], left: int = 0,
    batch_size: int = 100, output_dir: str = "Dataset/Encoded/WikiHow", skip_existing: bool = True,
):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    n_articles = len(dataset)

    batch_ranges = [
        (start, min(start + batch_size, n_articles))
        for start in range(left, n_articles, batch_size)
    ]

    failed_batches = []

    for batch_left, batch_right in tqdm(batch_ranges, desc="Encoding batches"):
        with suppress_all_output():
            final_file = output_path / f"encoded_WikiHow_{batch_left}_{batch_right}_vol2.pkl"
            temp_file = output_path / f"encoded_WikiHow_{batch_left}_{batch_right}_vol2.tmp"
            error_file = output_path / f"encoded_WikiHow_{batch_left}_{batch_right}_ERROR.txt"

            if skip_existing and final_file.exists():
                continue

            try:
                encoded_data, errors, n_qubits_list = preprocess_and_encode(
                    dataset[batch_left:batch_right]
                )

                encoded = {
                    "encoded_dataset": encoded_data,
                    "errors": errors,
                    "n_qubits": n_qubits_list,
                }

                # Write temporary file first
                with open(temp_file, "wb") as file:
                    pickle.dump(encoded, file, protocol=pickle.HIGHEST_PROTOCOL)

                # Rename only after successful write
                temp_file.replace(final_file)

                # Remove old error file if this batch succeeds later
                if error_file.exists():
                    error_file.unlink()

            except Exception as e:
                failed_batches.append((batch_left, batch_right))

                error_message = (
                    f"Batch failed: {batch_left}-{batch_right}\n"
                    f"Exception type: {type(e).__name__}\n"
                    f"Exception message: {e}\n\n"
                    f"Traceback:\n{traceback.format_exc()}"
                )

                with open(error_file, "w", encoding="utf-8") as file:
                    file.write(error_message)


            finally:
                gc.collect()

    summary = {
        "n_articles_requested": n_articles,
        "batch_size": batch_size,
        "n_batches": len(batch_ranges),
        "n_failed_batches": len(failed_batches),
        "failed_batches": failed_batches,
    }

    summary_file = output_path / "encoding_summary.pkl"
    with open(summary_file, "wb") as file:
        pickle.dump(summary, file, protocol=pickle.HIGHEST_PROTOCOL)

    print("Encoding finished.")
    print(f"Failed batches: {len(failed_batches)}")

    return summary

In [ ]:
dataset_full = load_WikiHow_JSON(FILE_PATH)
dataset = extract_relevant_data(dataset_full)

In [ ]:
from IPython.display import clear_output

In [ ]:
encode_auto(dataset, batch_size=100, left=2890)

In [ ]:
encoded_data, errors, n_qubits_list = preprocess_and_encode(dataset[95:110])

In [ ]:
# for i, error_message in enumerate(loaded_encoded_ds["errors"]):
for i, error_message in enumerate(errors):
    # if "will_train()" not in error_message and "quantum_encode()" not in error_message and "normalize()" not in error_message:
    # if "bytes" not in error_message and "quantum_encode()" not in error_message and "normalize()" not in error_message:
    if "will_train()" in error_message and "Too many" not in error_message:
        print(f"{i}: {error_message}")

In [ ]:
# np.mean(n_qubits_list)
a = []
for article in n_qubits_list:
    a.append(np.mean(article))

print(np.mean(a))

In [ ]:
def compare_to_pruned(ds):
    n_printed = 0
    for article in ds:
        pruned_sents, _ = sentence_simplify_spacy(article["text_sentences"])
        for i, sent in enumerate(article["text_sentences"]):
            if n_printed == 4:
                break
            print(f"{sent}\n{pruned_sents[i]}\n")
            n_printed += 1
        else:
            continue
        break

def get_length (dataset):
    s = 0
    l = 0
    for article in dataset:
        s += len(article["text_sentences"])
        l += len(article["labels"])
    # print(f"labels: {len(dataset)} | sentences: {s} | labels: {l}")
    return s, l

def get_length_encoded (dataset):
    s = 0
    l = 0
    c = 0
    for article in dataset:
        c += len(article["circuits"])
        l += len(article["labels"])
        s += len(article["original_text_sentences"])
    
    return s, l, c

def print_ratio(dataset, encoded_dataset):
    s, l = get_length(dataset)
    se, le, ce = get_length_encoded(encoded_dataset)

    print(f"labels: {len(dataset)} | sentences: {s} | labels: {l}")
    print(f"labels: {len(encoded_dataset)} | sentences: {se} | labels: {le} | circuits: {ce}")
    print(f"ratio: {se/s:0.2f}")

print_ratio(dataset, encoded_data)

In [ ]:
compare_to_pruned([dataset[2]])

In [ ]:
# labeled ds structure
full_record_of_labeled_article = {
    "article_id": ...,
    "n_sentences": ...,
    "n_positive": ...,
    "labeling_method": ...,
    "z_threshold": ,
    "sentences_data": scored = [{
                "sentence_id": ...,
                "sentence": ...,
                "score": ...,
                "label": ...,
                "label_vector": ...
            }]
}

# prefered structure of a prepered ds for encoding
record_of_article_to_encode = {
    "article_id": ,
    # "n_sentences": ..., # nereikia nes tsg galima len("sentences_data")
    # "n_positive": ..., # nereikia nes tsg galima .count("label[0,1]")
    "sentences_data": scored = [{
            "sentence_id": ...,
            "org_sentence": ...,
            "label_vector": ...,
            "extractive_score": ...,
            "circuit": ...,
            "n_circuit_qubits": ...,
            "n_circuit_gates": ...,
            "circuit_depth": ...,
        }]
}

# prefered structure
record_of_encoded_article = {
    "article_id": str,
    "sentence_ids": List[int],

    "sentences": List[str],
    "extractive_scores": List[float],

    "label_vectors": List[List[int]],
    "diagrams": List[lambeq.backend.grammar.Diagram], # before .to_tk() # but not sure if I will ever use this.
    "circuits": List[lambeq.backend.quantum.Diagram],

    "circuit_stats": {
        "n_qubits": List[int],
        "depth": List[int],
        "n_gates": List[int],
    }
}

# structure I sticked with
encoded_batch = {
    "encoded_dataset": encoded_data,
    "errors": List[List[errors]],
    "n_qubits": List[List[int]]
}

encoded_article = {
    "article_id": int,
    "circuits": List[lambeq.backend.quantum.Diagram],
    "labels": List[List[int]],
    "n_qubits": List[int],
    "original_text_sentences": List[str],
}

In [ ]:
ld_ds = load_PreSumm_pts(file_number=0, ds_purpose="train", calculate_articles=True)

In [ ]:
def encode_save_auto(dataset: List[Dict], n_articles: int, left: int=0):
    right = 0
    while right < n_articles:
        right = min(left + 100, n_articles)
        print(left, "-", right)
        encoded_data, errors, n_qubits_list = preprocess_and_encode(dataset[left:right])

        encoded = {
            "encoded_dataset": encoded_data,
            "errors": errors,
            "n_qubits": n_qubits_list
        }
        with open(f"Dataset/Encoded/cnn_dailymail/PreSumm_{left}_{right}.pkl", 'wb') as file:
            pickle.dump(encoded, file)

        left = right

In [ ]:
encode_save_auto(ld_ds, n_articles, left=100)

In [ ]:
# 14m 27s for the first 101 | without compiled circuit int will_train() | 28 | 200 | 20000

# 101 - 201
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [33:47<00:00, 20.28s/it]
# 201 - 301
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [31:48<00:00, 19.09s/it]
# 301 - 401
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [33:41<00:00, 20.21s/it]
# 401 - 501
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [31:08<00:00, 18.69s/it]
# 501 - 601
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [35:07<00:00, 21.08s/it]
# 601 - 701
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [32:48<00:00, 19.68s/it]

In [ ]:
def load_encoded_PreSumm_n_m(starting_index = 0, ending_index = 701):
    datasets_list = []
    for i in range(starting_index, ending_index, 100):
        with open(f"Dataset/Encoded/cnn_dailymail/PreSumm_{i}_{i + 100}.pkl", 'rb') as file:
            PreSum_loaded_ds = pickle.load(file)
            datasets_list.append(PreSum_loaded_ds)
    return datasets_list

In [ ]:
# print(get_deep_type(encoded_dataset))

encoded_valid_labels = remove_negative_articles(loaded_encoded_ds["encoded_dataset"], "circuits", "labels", [0,1])

In [ ]:
# PreSum_0_100.keys()

for i, error_message in enumerate(loaded_encoded_ds["errors"]):
    # if "will_train()" not in error_message and "quantum_encode()" not in error_message and "normalize()" not in error_message:
    if "bytes" not in error_message and "quantum_encode()" not in error_message and "normalize()" not in error_message:
        print(f"{i}: {error_message}")

In [ ]:
file_path = "Dataset/Raw/cnn_dailymail/_PreSumm/cnndm.test.0.bert.pt"
loaded_data = torch.load(file_path)

In [ ]:
import gc

# Delete large temporary variables
# del expensive_tensors

# Force Python to find unreferenced objects
gc.collect()

# Force the GPU to release the cached memory pool
torch.cuda.empty_cache()

In [ ]:
s = ["The president, speaking in Paris, announced sanctions while addressing the media.", 
     "My friend, a well-known scientist, published a paper.", 
     "The book that I read yesterday was fascinating.", 
     "He left the room while talking on the phone.", 
     "The CEO, who was under pressure, resigned after speaking to the board.", 
     "The president of the company announced reforms.", 
     "The cat sat on the mat."] 

expected_result = ["The president announced sanctions", "My friend published a paper", "The book was fascinating", "He left the room", "The CEO resigned", "The president of the company announced reforms", "The cat sat on the mat"]

for i, result in enumerate(sentence_simplify_spacy(s)[0]):
    print(s[i])
    print(result)
    print(expected_result[i], "\n")

In [ ]:
# s = combine_n_articles(ld_ds[:1], 1)[0]
s = ld_ds[:1][0]['text_sentences']
for i, result in enumerate(sentence_simplify_spacy(s)[0]):
    print(result)
    print(s[i], "\n")